In [2]:
# Load the golden dataset
import pandas as pd
df = pd.read_csv('../data/processed/golden_dataset.csv', parse_dates=['order_purchase_timestamp'])

# Snapshot date = last date in the dataset (NOT today)
snapshot = df['order_purchase_timestamp'].max()
print('Snapshot date:', snapshot)

# Recency — days since last purchase
recency_df = df.groupby('customer_unique_id', as_index=False).agg(
    last_purchase=('order_purchase_timestamp', 'max')
)
recency_df['recency'] = (snapshot - recency_df['last_purchase']).dt.days
recency_df['recency'] = recency_df['recency'].apply(lambda x: 1 if x == 0 else x)

# Frequency — unique orders per customer
freq_df = df.groupby('customer_unique_id', as_index=False).agg(
    frequency=('order_id', 'nunique')
)

# Monetary — total spend per customer
mon_df = df.groupby('customer_unique_id', as_index=False).agg(
    monetary=('payment_value', 'sum')
)

# Combine
rfm = recency_df.merge(freq_df, on='customer_unique_id')
rfm = rfm.merge(mon_df, on='customer_unique_id').drop(columns='last_purchase')
print(rfm.head())

Snapshot date: 2018-08-29 15:00:37
                 customer_unique_id  recency  frequency  monetary
0  0000366f3b9a7992bf8c76cfdf3221e2      111          1    141.90
1  0000b849f77a49e4a4ce2b2a4ca5be3f      114          1     27.19
2  0000f46a3911fa3c0805444483337064      536          1     86.22
3  0000f6ccb0745a6a4b88665a16c9f078      320          1     43.62
4  0004aac84e0df4da2b147fca70cf8255      287          1    196.89


In [3]:
# Score 1–5 using quintiles
rfm['R_score'] = pd.qcut(rfm['recency'], q=5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['monetary'], q=5, labels=[1,2,3,4,5])
rfm['RFM_score'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

# Map to business segments
def rfm_segment(row):
    r = int(str(row['RFM_score'])[0])
    m = int(str(row['RFM_score'])[2])
    if r >= 4 and m >= 4:
        return 'Champion'
    elif r >= 3 and m >= 3:
        return 'Loyal'
    elif r >= 3 and m <= 2:
        return 'Potential'
    elif r >= 2:
        return 'At-Risk'
    else:
        return 'Lost'

rfm['segment'] = rfm.apply(rfm_segment, axis=1)
print(rfm['segment'].value_counts())
rfm.to_csv('../data/processed/rfm_scores.csv', index=False)

segment
Potential    21976
Loyal        18728
Lost         18638
At-Risk      18576
Champion     15432
Name: count, dtype: int64
